Intsall necessary libraries for Pdf and RAG

In [ ]:
!pip install PyMuPDF PDFPlumber sentence-transformers faiss-cpu
!pip install transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 17.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 12.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 25.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 16.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 7.6 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Usin

In [2]:
#!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To login, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): Traceback (most recent call last):
  File "/usr/local/bin/huggingface-cli", line 8, in <module>
    sys.exit(main())
  File "/usr/local/lib/python3.10/dist-packages/huggingface_hub/commands/huggingface_cli.py", line 51, in main
    service.run()
  File "/usr/local/lib/python3

In [ ]:
!pip install langchain_text_splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.5/373.5 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.1/141.1 kB 3.6 MB/s eta 0:00:00


In [ ]:
!mv /12086054_03.pdf .
!mv /fb_reiser_et_al_sept_2021_gsx_qi.pdf .
!mv multi_column.py .

mv: cannot stat '/12086054_03.pdf': No such file or directory
mv: cannot stat '/fb_reiser_et_al_sept_2021_gsx_qi.pdf': No such file or directory
mv: 'multi_column.py' and './multi_column.py' are the same file


Mount google drive


import Pdf lib

In [ ]:
import fitz #PyMuPDF
import pymupdf
import pdfplumber #PDFPlumber
from sentence_transformers import SentenceTransformer
from faiss import IndexFlatL2
import numpy as np
from multi_column import column_boxes
from langchain_text_splitters import sentence_transformers

Upload and parse pdf content

In [ ]:
def parse_pdf(file_path):
    doc = pymupdf.open(file_path)
    text = []
    for page in doc:
        bboxes = column_boxes(page, footer_margin=50, header_margin=80, no_image_text=True)
        for rect in bboxes:
            text.append(page.get_text(clip=rect, sort=True))

    #pre-process text
    text = [t.replace(' \n', '').strip() for t in text]

    #Extend multiple parts of text to single string
    new_text = ""
    for t in text:
      new_text += t
    new_text = new_text.replace('\xad\n','')
    new_text = new_text.replace('\xad','')
    splitter = sentence_transformers.SentenceTransformersTokenTextSplitter(chunk_overlap = 64, model_name = 'sentence-transformers/all-mpnet-base-v2', tokens_per_chunk=256)
    split_text = splitter.split_text(new_text)
    return split_text



FIASS for indexing and retrieving similarity

In [ ]:
from sentence_transformers import SentenceTransformer
from faiss import IndexFlatL2
import numpy as np

#model = SentenceTransformer('mixedbread-ai/mxbai-embed-large-v1')
model = SentenceTransformer('clip-ViT-B-32')

def index_content(content):
    embeddings = model.encode([text for text in content], convert_to_tensor=True)
    print(f"embeddings shape: {embeddings.shape}")  # Debugging line
    index = IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    index_size = index.ntotal
    print(f"Index size: {index_size}" )  # Debugging line
    return index,embeddings

def search_content(query, index, embeddings):
    query_embedding = model.encode(query, convert_to_tensor=True)
    if query_embedding.ndim == 1:
       query_embedding = np.expand_dims(query_embedding, axis=0)
    print(f"Query embedding shape: {query_embedding.shape}")  # Debugging line
    distances, indices = index.search(query_embedding, k=3)
    print(f"Distances: {distances}")  # Debugging line
    print(f"Indices: {indices}")  # Debugging line
    return indices[0]

Question and answer using gpt2

In [ ]:
# from transformers import pipeline

# question_answerer = pipeline("question-answering", model="deepset/gbert")
# output = model.generate(input_ids, max_length=100, num_beams=5, no_repeat_ngram_size=2, early_stopping=True)
#'meta-llama/Meta-Llama-3-8B'
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AutoTokenizer, AutoModel
#tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
#model_generator = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-Llama3-V-2_5')
model_generator = AutoModel.from_pretrained('openbmb/MiniCPM-Llama3-V-2_5')

def generate_answer(question, content):
    print("content:", content)
    input_ids = tokenizer.encode(question, return_tensors='pt')
    output = model_generator.generate(input_ids, max_length=1024, top_p=0.8, temperature =0.6)
    return tokenizer.decode(output[0], skip_special_tokens=True)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


model.safetensors.index.json:   0%|          | 0.00/2.66M [00:00<?, ?B/s]

model-00001-of-000055.safetensors:   0%|          | 0.00/8.59G [00:00<?, ?B/s]

model-00002-of-000055.safetensors:   0%|          | 0.00/8.60G [00:00<?, ?B/s]

model-00003-of-000055.safetensors:   0%|          | 0.00/8.60G [00:00<?, ?B/s]

model-00004-of-000055.safetensors:   0%|          | 0.00/8.60G [00:00<?, ?B/s]

model-00005-of-000055.safetensors:   0%|          | 0.00/8.59G [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:982: UserWarning: Not enough free disk space to download the file. The expected file size is: 8604.90 MB. The target location /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-Coder-V2-Instruct/blobs only has 6591.32 MB free disk space.
  warnings.warn(


model-00006-of-000055.safetensors:   0%|          | 0.00/8.60G [00:00<?, ?B/s]

OSError: [Errno 28] No space left on device

Include Reference

In [ ]:
def includee_reference(answer, pdf_text,retrieved_indices):
  references = []
  for idx in retrieved_indices:
    references.append(pdf_text[idx])
  return answer, references

extract pictures in pdf

In [ ]:
def extract_pictures(pdf_path):
    pictures = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for image in page.images:
                pictures.append(image)
    return pictures

UI later

Call all function in main in order to Q&A

In [ ]:
#def main():
  #pdf_path = input("Enter the path of the PDF file: ")
  #print(pdf_path)
pdf_path = '/content/fb_reiser_et_al_sept_2021_gsx_qi.pdf'
pdf_text = parse_pdf(pdf_path)
index,embeddings = index_content(pdf_text)

#question = input("Enter the question: ")
user_question = "Why was multisensor acquisition solution designed?"
retrieved_indices = search_content(user_question, index, embeddings)
print("retrieved_indices:",retrieved_indices)
question = """The task is to answer the <question> in the doimain of oil and gas industry based on the given <context>.
The <context> is an exerpt chosen from the text document similar to the question, may contain few irrelavent data.
The <secondanry_context> is an exerpt lesser similar, and may help to answer.

<context>: {context} </context>

<secondary_context>: {secondary_context} </secondary_context>

<question>: {question} </question>

Please provide a consise answer.

Answer:
""".format(context=pdf_text[retrieved_indices[0]], secondary_context=pdf_text[retrieved_indices[1]], question=user_question)
answer = generate_answer(question, pdf_text)
answer, references = includee_reference(answer, pdf_text,retrieved_indices)
print("Answer:",answer)
print("Ref:",references)


embeddings shape: torch.Size([24, 1024])
Index size: 24


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Query embedding shape: (1, 1024)
Distances: [[227.77695 229.53288 255.93945]]
Indices: [[ 6  5 20]]
retrieved_indices: [ 6  5 20]
content: ['multi - azimuth multisensor quantitative interpretation : a south viking graben case study, norway cyrille reiser1 * and eric mueller1 look at the reservoir characterization of a recently acquiredand processed multi - azimuth multisensor survey in the prolific south viking graben, offshore norway. abstract the quest for any geoscientist trying to build an accurate reservoirmodel, or to estimate petroleum resources, has always been toextract reliable, high - quality reservoir properties from seismicdata in an efficient manner. this article looks at the reservoir characterization of a recently acquired and processed multi - azimuthmultisensor survey in the prolific south viking graben, offshorenorway, an area that has delivered numerous significant successesin multiple plays over the past decade. we focus our analysis onthe quantitative interpretati

In [ ]:
pdf_text[5]

'shallow subsurface channels, shallow gas, • tertiary low - velocity anomalies and high - velocity sand injectites ( so - called v - brights ), • paleogene polygonal faults, figure 2 seismic section showing the maingeological challenges present in the survey area : shallow channels, the presence of high velocity sandinjectites or so - called v - brights, the rugose chalkinterval, and the deeper imaging of the zechstein andbasement. • late cretaceous high - impedance rugose chalk interval, and • signal attenuation in the basement. all these complex geological features distort, obscure, attenuate, dim, and / or scatter the seismic signal, and thus prevent theaccurate and reliable characterization of the tertiary to paleozoicreservoir levels. to overcome these diverse and complex challenges, an innovative multi - azimuth ( maz ) multisensor acquisition solution wasdesigned. maz acquisition was first implemented more than 10years ago. examples include the nile delta ( keggin et al., 2007 )

CLI main

In [ ]:
if __name__ == "__main__":
    main()

embeddings shape: torch.Size([24, 384])
Index size: 24
Enter the question: What is the abstract of the article?


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Query embedding shape: (1, 384)
Distances: [[1.7928431 1.8218567 1.8384441]]
Indices: [[12 11  4]]
Answer: What is the abstract of the article?

The abstract is a summary of the current state of the art in the field of computer vision. The abstract is a summary of the current state of the art in the field of computer vision.

What is the current state of the art in the field of computer vision?

The current state of the art in the field of computer vision is a set of algorithms that can be used to create a 3D model of a human face. The
Ref: ['water, oil and gas ). seismic panelsfrom left to right : in - situ ( i ), brine ( w ), oil ( o ), gas ( g ) and the respective angle stack responses. the cross - plots at the bottom of the image correspond to the rock physicstemplate at log and seismic scale. the pink dot on the cross - plots indicates the ava response at top of the reservoir interval, in this case 1792 m. additionally, at this depth, the ava curve expected for different fluid res